<a href="https://colab.research.google.com/github/towardsai/agentic-ai-engineering-course/blob/main/lessons/11_multimodal/notebook_exercise.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lesson 11: Multimodal

This notebook demonstrates how to build multimodal AI systems that can process and understand multimodal data such as text, images and documents using Google's Gemini models.

We will use the `google-genai` library to interact with Google's Gemini models.

**Learning Objectives:**

1. **Process multimodal content**: Learn to handle images and PDFs in different formats (bytes, base64, URLs) with Gemini models
2. **Implement object detection**: Use multimodal LLMs for visual analysis and structured output generation
3. **Build multimodal RAG systems**: Create and index embeddings for images, documents and text to enable semantic search across multimodal content
4. **Develop multimodal AI agents**: Construct ReAct agents that can search through and reason about multimodal information

> **Exercise version.** This is the exercise notebook for Lesson 11. The full solution lives in [`notebook.ipynb`](https://colab.research.google.com/github/towardsai/agentic-ai-engineering-course/blob/main/lessons/11_multimodal/notebook.ipynb) in the same folder. Attempt each exercise before checking the solutions.

### Exercise roadmap

| Exercise | Difficulty | What you build |
|---|---|---|
| 1 | Intermediate | The image loader that resizes and converts to bytes |
| 2 | Starter | The base64 encoding variant of the loader |
| 3 | Intermediate | Object detection with a multimodal structured-output call |
| 4 | Intermediate | The multimodal vector index over raw image bytes |
| 5 | Advanced | Cross-modal semantic search over the index |
| 6 | Starter | The multimodal ReAct agent assembly |

## 1. Setup

### Set Up Python Environment

**Google Colab:** Run the code cell below — it installs all required packages and loads your `GOOGLE_API_KEY` from Colab Secrets automatically.

To set up your Python virtual environment using `uv` and load it into the Notebook, follow the step-by-step instructions from the `Course Admin` lesson from the beginning of the course.

**TL;DR:** Be sure the correct kernel pointing to your `uv` virtual environment is selected.

In [ ]:
# ============================================================
# Google Colab Setup — runs only when executed in Colab
# ============================================================
import sys

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    import importlib
    import os
    import site
    import subprocess

    # Install the course package (published from pyproject.toml) and its pinned extras
    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "-U",
            "agentic-ai-engineering-course==0.4.8",
            "nest-asyncio2",
            "google-auth==2.53.0",
            "opentelemetry-api==1.42.1",
            "opentelemetry-sdk==1.42.1",
            "opentelemetry-exporter-otlp-proto-http==1.42.1",
            "opentelemetry-exporter-otlp-proto-common==1.42.1",
            "opentelemetry-proto==1.42.1",
            "jedi==0.18.2",
        ],
        check=True,
    )
    importlib.reload(site)  # make newly installed packages importable without restart

    # Load API key from Colab Secrets
    # In Colab: Secrets tab (key icon) → Add new secret → Name: GOOGLE_API_KEY
    from google.colab import userdata

    os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")

    # Download lesson assets (images and PDFs) from the course repo
    os.makedirs("images", exist_ok=True)
    os.makedirs("pdfs", exist_ok=True)

    BASE = "https://github.com/towardsai/agentic-ai-engineering-course/raw/main/lessons/11_multimodal"
    assets = [
        ("images/attention_is_all_you_need_0.jpeg", f"{BASE}/images/attention_is_all_you_need_0.jpeg"),
        ("images/attention_is_all_you_need_1.jpeg", f"{BASE}/images/attention_is_all_you_need_1.jpeg"),
        ("images/attention_is_all_you_need_2.jpeg", f"{BASE}/images/attention_is_all_you_need_2.jpeg"),
        ("images/image_1.jpeg", f"{BASE}/images/image_1.jpeg"),
        ("images/image_2.jpeg", f"{BASE}/images/image_2.jpeg"),
        ("images/image_3.jpeg", f"{BASE}/images/image_3.jpeg"),
        ("images/image_4.jpeg", f"{BASE}/images/image_4.jpeg"),
        ("pdfs/attention_is_all_you_need_paper.pdf", f"{BASE}/pdfs/attention_is_all_you_need_paper.pdf"),
    ]
    for dest, url in assets:
        subprocess.run(["wget", "-q", "-O", dest, url], check=True)

In [ ]:
if not IN_COLAB:
    get_ipython().run_line_magic("load_ext", "autoreload")
    get_ipython().run_line_magic("autoreload", "2")

    from utils import env

    env.load(required_env_vars=["GOOGLE_API_KEY"])

### Import Key Packages

In [ ]:
import base64
import io
from pathlib import Path
from typing import Literal

from google import genai
from google.genai import types
from IPython.display import Image as IPythonImage
from PIL import Image as PILImage
from utils import pretty_print

### Initialize the Gemini Client

In [ ]:
client = genai.Client()

### Define Constants

We will use the `gemini-3.5-flash` model, which is fast and cost-effective:

In [ ]:
MODEL_ID = "gemini-3.5-flash"

## 2. Applying Multimodal LLMs to Images and PDFs

There are three core ways we can process images and PDFs with multimodal LLMs:
1. As raw bytes
2. As base64 encoded strings
3. As URLs

We will first look into how we can process images and then PDFs.

Now, let's look at our test image:


In [ ]:
def display_image(image_path: Path) -> None:
    """
    Display an image from a file path in the notebook.

    Args:
        image_path: Path to the image file to display

    Returns:
        None
    """

    image = IPythonImage(filename=image_path, width=400)
    display(image)


display_image(Path("images") / "image_1.jpeg")

### 2.1 As Raw Bytes

### Exercise 1: Load images as bytes

Every multimodal call in this lesson starts here: turning an image file into resized, compressed bytes. Smaller images mean fewer tokens and faster calls, so the resize step is not cosmetic.

**Learning goal:** Prepare image files as bytes suitable for a multimodal LLM call.

**What you need to implement:**

1. Open the image from the given path
2. If its width exceeds `max_width`, resize it proportionally (compute the ratio from the widths and scale the height by the same ratio)
3. Save the image into an in-memory byte stream using the requested format
4. Return the bytes, or a (bytes, size) tuple when `return_size` is true

**Key concepts:**

- `PILImage.open()` loads an image, `.width`/`.height` expose its dimensions, `.resize()` takes a (width, height) tuple
- `io.BytesIO()` is an in-memory file: save into it, then read back with `.getvalue()`
- `image.size` after resizing gives the (width, height) the detection exercises later need for visualization

**Expected output:** the demo cell below prints the first bytes of a WEBP buffer and its size, roughly a few tens of KB for the kitten photo.

**Implementation hints:**

- Resize before saving, the whole point is sending fewer pixels over the wire
- Every captioning and detection cell below depends on this function, they will error or fail until it returns real bytes

In [ ]:
# === Exercise cell: fill in the gaps below ===

def load_image_as_bytes(
    image_path: Path, format: Literal["WEBP", "JPEG", "PNG"] = "WEBP", max_width: int = 600, return_size: bool = False
) -> bytes | tuple[bytes, tuple[int, int]]:
    """
    Load an image from file path and convert it to bytes with optional resizing.

    Args:
        image_path: Path to the image file to load
        format: Output image format (WEBP, JPEG, or PNG). Defaults to "WEBP"
        max_width: Maximum width for resizing. If image width exceeds this, it will be resized proportionally. Defaults to 600
        return_size: If True, returns both bytes and image size tuple. Defaults to False

    Returns:
        bytes: Image data as bytes, or tuple of (bytes, (width, height)) if return_size is True

    Steps to complete:
    1. Open the image
    2. Resize proportionally when the width exceeds max_width
    3. Save it into an in-memory byte stream with the requested format
    4. Return the bytes (plus the image size when return_size is True)
    """
    # Your implementation goes here

    if return_size:
        return b"", (0, 0)  # Replace with the real bytes and size
    return b""  # Replace with the real image bytes

Load image:

In [ ]:
image_bytes = load_image_as_bytes(image_path=Path("images") / "image_1.jpeg", format="WEBP")
pretty_print.wrapped([f"Bytes `{image_bytes[:30]}...`", f"Size: {len(image_bytes)} bytes"], title="Image as Bytes")

Compute captions:

In [ ]:
response = client.models.generate_content(
    model=MODEL_ID,
    contents=[
        types.Part.from_bytes(
            data=image_bytes,
            mime_type="image/webp",
        ),
        "Tell me what is in this image in one paragraph.",
    ],
)
pretty_print.wrapped(response.text, title="Image 1 Caption")

Using the same approach, we can easily pass multiple images simultaneously. For example, the previous one plus the one below, and compare them:

In [ ]:
display_image(Path("images") / "image_2.jpeg")

In [ ]:
response = client.models.generate_content(
    model=MODEL_ID,
    contents=[
        types.Part.from_bytes(
            data=load_image_as_bytes(image_path=Path("images") / "image_1.jpeg", format="WEBP"),
            mime_type="image/webp",
        ),
        types.Part.from_bytes(
            data=load_image_as_bytes(image_path=Path("images") / "image_2.jpeg", format="WEBP"),
            mime_type="image/webp",
        ),
        "What's the difference between these two images? Describe it in one paragraph.",
    ],
)
pretty_print.wrapped(response.text, title="Differences between images")

### 2.2 As Base64 Encoded Strings

Now, let's load the same image as base64:

### Exercise 2: Encode images as base64

Some APIs and storage layers want text-safe payloads instead of raw bytes. Base64 is that encoding, at the price of roughly 33% more volume.

**Learning goal:** Convert an image to a base64 string by reusing the bytes loader.

**What you need to implement:**

1. Get the image bytes by delegating to `load_image_as_bytes` with the same path, format, and max width (no size tuple needed)
2. Base64-encode the bytes and decode the result to a UTF-8 string

**Key concepts:**

- `base64.b64encode()` returns bytes, the trailing decode step is what turns it into a `str`
- `cast(bytes, ...)` keeps the type checker happy since the loader's return type is a union

**Expected output:** the demo below prints a base64 prefix and a character count roughly a third larger than the byte count from Exercise 1.

**Implementation hints:**

- This is a two-step delegation, if you are writing PIL code here, step back
- The size-comparison and caption cells below need both exercises done to make sense

In [ ]:
# === Exercise cell: fill in the gaps below ===

from typing import cast


def load_image_as_base64(
    image_path: Path, format: Literal["WEBP", "JPEG", "PNG"] = "WEBP", max_width: int = 600, return_size: bool = False
) -> str:
    """
    Load an image and convert it to base64 encoded string.

    Args:
        image_path: Path to the image file to load
        format: Output image format (WEBP, JPEG, or PNG). Defaults to "WEBP"
        max_width: Maximum width for resizing. If image width exceeds this, it will be resized proportionally. Defaults to 600
        return_size: Parameter passed to load_image_as_bytes function. Defaults to False

    Returns:
        str: Base64 encoded string representation of the image

    Steps to complete:
    1. Load the image bytes with load_image_as_bytes (same path, format, and
       max width, without the size tuple)
    2. Base64-encode those bytes and decode them to a UTF-8 string
    """
    # Your implementation goes here

    return ""  # Replace with the base64 string

In [ ]:
image_base64 = load_image_as_base64(image_path=Path("images") / "image_1.jpeg", format="WEBP")
pretty_print.wrapped([f"Base64: {image_base64[:100]}...`", f"Size: {len(image_base64)} characters"], title="Image as Base64")

On average base64 format is 33% larger than raw bytes. As we can see in this use case as well:

In [ ]:
print(f"Image as Base64 is {(len(image_base64) - len(image_bytes)) / len(image_bytes) * 100:.2f}% larger than as bytes")

Now, let's recompute the image caption using this method:

In [ ]:
response = client.models.generate_content(
    model=MODEL_ID,
    contents=[
        types.Part.from_bytes(data=image_base64, mime_type="image/webp"),
        "Tell me what is in this image in one paragraph.",
    ],
)
response.text

### 2.3 As Public URLs

Using Gemini's `url_context` out-of-the-box tool, we can automatically visit and parse webpages, PDFs, and images from the open internet. You only have to provide the direct URL in the prompt and configure the `url_context` tool. This makes it a no-brainer to parse multiple data formats when available online:

In [ ]:
response = client.models.generate_content(
    model=MODEL_ID,
    contents="Based on the provided paper as a PDF, tell me how ReAct works: https://arxiv.org/pdf/2210.03629",
    config=types.GenerateContentConfig(tools=[{"url_context": {}}]),
)
pretty_print.wrapped(response.text, title="How ReAct works")

### 2.4 As URLs from Private Data Lakes

At the time of writing this notebook, Gemini works well primarily with GCP Cloud Storage links and not with other buckets such as S3. Buckets are excellent for production use cases, but they complicate our simple demonstration. Therefore, we will show you a mocked example.

The code would look like this, where you have to change the `uri` and ensure the LLM has the right permissions to your GCS bucket:
```python
response = client.models.generate_content(
    model=MODEL_ID,
    contents=[
        types.Part.from_uri(uri="gs://gemini-images/image_1.jpeg", mime_type="image/webp"),
        "Tell me what is in this image in one paragraph.",
    ],
)
```

### 2.5 Object Detection with LLMs

As a more exciting example, let's do object detection with multimodal LLMs.

First, let's define the output Pydantic models:

In [ ]:
from pydantic import BaseModel, Field


class BoundingBox(BaseModel):
    ymin: float
    xmin: float
    ymax: float
    xmax: float
    label: str = Field(default="The category of the object found within the bounding box. For example: cat, dog, diagram, robot.")


class Detections(BaseModel):
    bounding_boxes: list[BoundingBox]

Then the prompt and image:

In [ ]:
prompt = """
Detect all of the prominent items in the image. 
The box_2d should be [ymin, xmin, ymax, xmax] normalized to 0-1000.
Also, output the label of the object found within the bounding box.
"""

image_bytes, image_size = load_image_as_bytes(image_path=Path("images") / "image_1.jpeg", format="WEBP", return_size=True)

Now, let's call the LLM:

### Exercise 3: Detect objects with a structured multimodal call

Captioning returns prose. Detection returns coordinates. Combining an image part with a structured-output schema gives you typed bounding boxes straight from the model.

**Learning goal:** Combine multimodal input with native structured outputs in a single call.

**What you need to implement:**

1. Build a generation config that forces JSON output validated against the `Detections` schema
2. Call the model with two content entries: the image bytes as a WEBP part, and the detection `prompt` defined above
3. Cast the parsed response to `Detections` and store it in `detections`

**Key concepts:**

- `types.Part.from_bytes(data=..., mime_type=...)` wraps binary content for the request
- `response.parsed` holds the schema-validated object, `cast(Detections, ...)` narrows its type
- The prompt's instruction that boxes are normalized to 0-1000 is what makes the visualization math below work

**Expected output:** a "Detections" block listing the image size and a handful of `BoundingBox` entries with labels like cat or robot.

**Implementation hints:**

- This is the Lesson 4 native-structured-output pattern with an image part added to the contents
- The visualization cell below reads `detections.bounding_boxes`, it will error until this cell produces a real `Detections` object

In [ ]:
# === Exercise cell: fill in the gaps below ===

# Steps to complete:
# 1. Build a generation config that forces JSON output validated against Detections
# 2. Call the model with the image bytes (as a WEBP part) and the prompt above
# 3. Cast the parsed response to Detections and store it in `detections`

config = None  # TODO 1: the structured-output config
response = None  # TODO 2: the multimodal model call
detections = None  # TODO 3: the parsed Detections object

# Your implementation goes here

# Smoke test
if detections is not None:
    pretty_print.wrapped([f"Image size: {image_size}", *detections.bounding_boxes], title="Detections")
else:
    print("(detections not set yet, complete the TODOs above)")

Let's also visualize the bounding boxes: 

In [ ]:
import matplotlib.patches as patches
import matplotlib.pyplot as plt
import numpy as np


def visualize_detections(detections: Detections, image_path: Path) -> None:
    """
    Visualize detected bounding boxes on an image with red rectangles and labels.

    Args:
        detections: Detections object containing bounding boxes in [ymin, xmin, ymax, xmax] format normalized to 0-1000
        image_path: Path to the image file to visualize

    Returns:
        None: Displays the image with bounding boxes in the notebook
    """

    # Clear any existing plots to prevent overlapping
    plt.clf()

    image = PILImage.open(image_path)
    image_array = np.array(image)
    img_height, img_width = image_array.shape[:2]

    fig, ax = plt.subplots(1, 1, figsize=(8, 6))
    ax.imshow(image_array)

    for bbox in detections.bounding_boxes:
        # Convert normalized coordinates (0-1000) to pixel coordinates
        xmin = (bbox.xmin / 1000) * img_width
        ymin = (bbox.ymin / 1000) * img_height
        xmax = (bbox.xmax / 1000) * img_width
        ymax = (bbox.ymax / 1000) * img_height

        # Calculate box dimensions (matplotlib uses bottom-left corner + width/height)
        width = xmax - xmin
        height = ymax - ymin

        # Create rectangle patch (x, y is bottom-left corner)
        rect = patches.Rectangle((xmin, ymin), width, height, linewidth=3, edgecolor="red", facecolor="none")

        # Add rectangle to the plot
        ax.add_patch(rect)

        # Add label text (positioned at top-left of bounding box)
        ax.text(
            xmin,
            ymin + 5,  # Slightly above the box
            bbox.label[:15],
            fontsize=12,
            color="red",
            fontweight="bold",
            bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.8),
        )

    # Remove axis ticks and labels for cleaner display
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_title(f"Object Detection Results: {image_path.name}", fontsize=14, fontweight="bold")

    plt.tight_layout()
    plt.show()

In [ ]:
visualize_detections(detections, Path("images") / "image_1.jpeg")

### 2.6 Working with PDFs

Ultimately, let's see how we can work with PDFs. We will use the legendary `Attention Is All You Need` paper as an example. 

To display it, we extracted the first 3 pages of the PDF as images. For example, this is how the page looks:


In [ ]:
display_image(Path("images") / "attention_is_all_you_need_0.jpeg")

We can treat PDFs similarly to images. Therefore, we can pass PDFs as bytes:

In [ ]:
pdf_bytes = (Path("pdfs") / "attention_is_all_you_need_paper.pdf").read_bytes()
pretty_print.wrapped(f"Bytes: {pdf_bytes[:40]}...", title="PDF bytes")

Call the LLM:

In [ ]:
response = client.models.generate_content(
    model=MODEL_ID,
    contents=[
        types.Part.from_bytes(data=pdf_bytes, mime_type="application/pdf"),
        "What is this document about? Provide a brief summary of the main topics.",
    ],
)
pretty_print.wrapped(response.text, title="PDF Summary (as bytes)")

Alternatively, as base64 encoded strings:

In [ ]:
def load_pdf_as_base64(pdf_path: Path) -> str:
    """
    Load a PDF file and convert it to base64 encoded string.

    Args:
        pdf_path: Path to the PDF file to load

    Returns:
        str: Base64 encoded string representation of the PDF
    """

    with open(pdf_path, "rb") as f:
        return base64.b64encode(f.read()).decode("utf-8")

Load the PDF:

In [ ]:
pdf_base64 = load_pdf_as_base64(pdf_path=Path("pdfs") / "attention_is_all_you_need_paper.pdf")
pretty_print.wrapped(f"Base64: {pdf_base64[:40]}...", title="PDF as Base64")

Call the LLM:

In [ ]:
response = client.models.generate_content(
    model=MODEL_ID,
    contents=[
        "What is this document about? Provide a brief summary of the main topics.",
        types.Part.from_bytes(data=pdf_base64, mime_type="application/pdf"),
    ],
)

pretty_print.wrapped(response.text, title="PDF Summary (as base64)")

Now, let's do a more interesting example and detect the diagrams from a page of the transformers paper, such as the one below:

In [ ]:
display_image(Path("images") / "attention_is_all_you_need_1.jpeg")

Define the object detection prompt to detect diagrams (similar to how we did for images):

In [ ]:
prompt = """
Detect all the diagrams from the provided image as 2d bounding boxes. 
The box_2d should be [ymin, xmin, ymax, xmax] normalized to 0-1000.
Also, output the label of the object found within the bounding box.
"""

image_bytes, image_size = load_image_as_bytes(
    image_path=Path("images") / "attention_is_all_you_need_1.jpeg", format="WEBP", return_size=True
)

Call the LLM:

In [ ]:
config = types.GenerateContentConfig(
    response_mime_type="application/json",
    response_schema=Detections,
)
response = client.models.generate_content(
    model=MODEL_ID,
    contents=[
        types.Part.from_bytes(
            data=image_bytes,
            mime_type="image/webp",
        ),
        prompt,
    ],
    config=config,
)
detections = cast(Detections, response.parsed)
pretty_print.wrapped([f"Image size: {image_size}", *detections.bounding_boxes], title="Detections")

Visualize the detections:

In [ ]:
visualize_detections(detections, Path("images") / "attention_is_all_you_need_1.jpeg")

## 3. Implementing Multimodal RAG for Images, PDFs and Text

To bring everything we did in this course together, let's implement a multimodal RAG system that works with text, images, and PDFs.

These are the images and PDF pages (as images) we will index for semantic search:

In [ ]:
def display_image_grid(
    image_paths: list[Path],
    titles: list[str] | None = None,
    rows: int | None = None,
    cols: int | None = None,
    figsize: tuple | None = None,
) -> None:
    """
    Display a grid of images, optionally captioned.

    Args:
        image_paths: List of paths to images to display.
        titles: Optional per-image captions, aligned with `image_paths`.
        rows: Number of rows. Auto-computed from `cols` when omitted.
        cols: Number of columns. Defaults to `min(3, len(image_paths))`.
        figsize: Figure size as (width, height). Auto-scaled to the grid when omitted.
    """

    if not image_paths:
        return

    cols = cols or min(3, len(image_paths))
    rows = rows or (len(image_paths) + cols - 1) // cols
    figsize = figsize or (4 * cols, 3.6 * rows)

    fig, axes = plt.subplots(rows, cols, figsize=figsize)
    axes = np.array(axes).ravel()
    for ax in axes:
        ax.axis("off")

    for idx, img_path in enumerate(image_paths[: rows * cols]):
        axes[idx].imshow(PILImage.open(img_path))
        if titles is not None:
            axes[idx].set_title(titles[idx], fontsize=9)

    plt.tight_layout()
    plt.show()


display_image_grid(
    image_paths=[
        Path("images") / "image_1.jpeg",
        Path("images") / "image_2.jpeg",
        Path("images") / "image_3.jpeg",
        Path("images") / "image_4.jpeg",
        Path("images") / "attention_is_all_you_need_1.jpeg",
        Path("images") / "attention_is_all_you_need_2.jpeg",
    ],
    rows=2,
    cols=3,
)

Now, let's define the core function that creates a multimodal embedding using [`gemini-embedding-2`](https://ai.google.dev/gemini-api/docs/embeddings), which is a multimodal embedding model from the Gemini family of models. [More here](https://ai.google.dev/gemini-api/docs/embeddings).

Unlike a text-only embedder, this model maps **text, images, audio, video and PDFs into the same embedding space**, so we can embed our images and PDFs **directly** as raw bytes, with no text-description proxy required:

In [ ]:
import asyncio
from typing import Any

import numpy as np

MimeType = Literal["image/webp", "image/png", "image/jpeg", "application/pdf", "audio/mpeg", "video/mp4"]


def embed(
    content: str | bytes,
    mime_type: MimeType | None = None,
    output_dimensionality: int | None = None,
) -> np.ndarray | None:
    """
    Embed text or raw bytes (image / PDF / audio / video) using Gemini's
    multimodal embedding model `gemini-embedding-2`.

    Args:
        content: Text string OR raw bytes.
        mime_type: Required when `content` is bytes (e.g. "image/webp",
            "application/pdf"). Ignored for text.
        output_dimensionality: Optional output size (128\u20133072). Defaults to
            the model default (3072).

    Returns:
        np.ndarray | None: Embedding vector or None if the call failed.
    """

    if isinstance(content, bytes):
        if mime_type is None:
            raise ValueError("mime_type is required when content is bytes")
        contents: list = [types.Part.from_bytes(data=content, mime_type=mime_type)]
    else:
        contents = [content]

    config = types.EmbedContentConfig(output_dimensionality=output_dimensionality) if output_dimensionality is not None else None

    try:
        kwargs: dict[str, Any] = {"model": "gemini-embedding-2", "contents": contents}
        if config is not None:
            kwargs["config"] = config
        result = client.models.embed_content(**kwargs)

        if not result or not result.embeddings:
            print("\u274c No embedding data found in response")
            return None

        return np.array(result.embeddings[0].values)

    except Exception as e:
        print(f"\u274c Failed to embed content: {e}")
        return None


async def embed_batch(
    items: list[tuple[str | bytes, MimeType | None]],
    output_dimensionality: int | None = None,
    max_concurrency: int = 8,
) -> list[np.ndarray | None]:
    """
    Embed many items concurrently by fanning out one async call per item.

    Each item is a `(content, mime_type)` tuple. Pass `mime_type=None` for text.
    The calls run concurrently (bounded by `max_concurrency`) against the stable
    `client.aio.models.embed_content` endpoint, and results are returned in the
    same order as the inputs.

    Note:
        The SDK also exposes `client.batches.create_embeddings`, which is cheaper
        (~half price) for very large, latency-tolerant jobs. We deliberately
        avoid it here because it is flagged **experimental** ("may change without
        notice") in the current `google-genai` release. Async fan-out over the
        stable single-item endpoint gives us the same "embed a whole list in one
        call" ergonomics on an API that is guaranteed not to break underneath
        us. For huge *offline* indexing jobs where cost matters more than
        stability, prefer the batch API once it stabilises.
        See https://ai.google.dev/gemini-api/docs/batch-api#batch-embedding

    Args:
        items: List of `(content, mime_type)` tuples.
        output_dimensionality: Optional Matryoshka truncation. See `embed`.
        max_concurrency: Maximum number of in-flight embedding requests.

    Returns:
        Embeddings in the same order as `items`. Failed items are `None`.
    """

    config = types.EmbedContentConfig(output_dimensionality=output_dimensionality) if output_dimensionality is not None else None
    semaphore = asyncio.Semaphore(max_concurrency)

    async def _embed_one(content: str | bytes, mime_type: MimeType | None) -> np.ndarray | None:
        if isinstance(content, bytes):
            if mime_type is None:
                raise ValueError("mime_type is required when content is bytes")
            contents: list = [types.Part.from_bytes(data=content, mime_type=mime_type)]
        else:
            contents = [content]

        kwargs: dict[str, Any] = {"model": "gemini-embedding-2", "contents": contents}
        if config is not None:
            kwargs["config"] = config

        async with semaphore:
            try:
                result = await client.aio.models.embed_content(**kwargs)
            except Exception as e:
                print(f"❌ Failed to embed content: {e}")
                return None

        if not result or not result.embeddings:
            print("❌ No embedding data found in response")
            return None

        return np.array(result.embeddings[0].values)

    return list(await asyncio.gather(*(_embed_one(content, mime_type) for content, mime_type in items)))

Let's first try it with a text input:

In [ ]:
embedding = embed("This is a test")
embedding

...and now embed an **image** with the *same* function (with no text proxy):

In [ ]:
image_bytes = load_image_as_bytes(image_path=Path("images") / "image_1.jpeg", format="WEBP")
image_embedding = embed(image_bytes, mime_type="image/webp")
image_embedding

Both calls return a `3072`-dim vector in the **same** embedding space, so we can compare a text query against an image embedding directly via semantic similarity:

In [ ]:
embedding.shape, image_embedding.shape

With a semantic similarity score of:

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

cosine_similarity([embedding], [image_embedding])

Let's now create the vector index over our test images and PDF pages, embedding each one **directly** as raw bytes, with no text proxy. We embed them with `embed_batch`, which fans the requests out concurrently over the stable async embedding endpoint, so the whole 7-image index is built with one batched call instead of many sequential round-trips:

### Exercise 4: Build the multimodal vector index

Here is the RAG turn: embed every image directly as raw bytes (no caption proxy) into the shared embedding space, and keep the bytes alongside the vectors so retrieved items can be handed straight back to a model.

**Learning goal:** Build a searchable index of multimodal embeddings with their source content.

**What you need to implement:**

1. Load every image as WEBP bytes (reuse your Exercise 1 loader, casting each result to bytes)
2. Embed all byte payloads concurrently with `embed_batch`, passing each as a (content, mime type) tuple
3. Zip paths, bytes, and embeddings into one dict per image with the five keys documented in the docstring

**Key concepts:**

- `embed_batch` is given above: it fans out async embedding calls and returns vectors in input order
- Storing `content` (the actual bytes) next to `embedding` is what lets the agent in section 5 re-read retrieved items
- The shared embedding space is the whole trick: these image vectors are directly comparable to text query vectors

**Expected output:** the check cell below reports 7 embeddings under `vector_index`, and the inspection cells show a dict with content/type/filename/mime_type/embedding and a 3072-dim vector.

**Implementation hints:**

- One list comprehension per step keeps this readable: bytes list, then the awaited batch, then the zip
- The index inspection cells below raise an IndexError while this returns an empty list

In [ ]:
# === Exercise cell: fill in the gaps below ===

from typing import cast


async def create_vector_index(image_paths: list[Path]) -> list[dict]:
    """
    Create a vector index over a collection of images by embedding each image
    **directly** with Gemini's multimodal embedding model, fanning the calls
    out concurrently with `embed_batch`.

    No intermediate text description is required — the embeddings live in the
    same shared space as text embeddings, so a text query can be compared
    directly against image embeddings.

    Args:
        image_paths: List of paths to image files to index.

    Returns:
        list[dict]: List of dictionaries with the following keys:
            - content (bytes): Raw image bytes
            - type (str): Always "image"
            - filename (Path): Original image path
            - mime_type (str): Always "image/webp"
            - embedding (np.ndarray): Multimodal embedding of the image bytes

    Steps to complete:
    1. Load each image as WEBP bytes with load_image_as_bytes
    2. Embed all payloads concurrently with embed_batch, one (bytes, mime type)
       tuple per image
    3. Zip paths, bytes, and embeddings into one dict per image with the keys
       documented above
    """
    # Your implementation goes here

    return []  # Replace with the list of index entries

We call the `create_vector_index` function on all the images from the `images` dir:

In [ ]:
image_paths = list(Path("images").glob("*.jpeg"))
vector_index = await create_vector_index(image_paths)

In [ ]:
if len(vector_index) == 0:
    pretty_print.wrapped("Could not create the vector index.", title="❌")
else:
    pretty_print.wrapped(f"Successfully created {len(vector_index)} embeddings under the `vector_index` variable", title="✅")

This is what an element of the `vector_index` looks like:

In [ ]:
vector_index[0]

In [ ]:
vector_index[0]["embedding"].shape

In [ ]:
print(f"type={vector_index[0]['type']!r}, filename={vector_index[0]['filename']}")

### Validation check - run this after your implementation

Uncomment the cell below and run it after building the index above. It inspects the in-memory `vector_index`, no API calls.

In [ ]:
# # Validation: check your Exercise 4 implementation
# try:
#     assert vector_index, "❌ vector_index is empty. Implement create_vector_index and re-run the indexing cell."
#     _required = {"content", "type", "filename", "mime_type", "embedding"}
#     assert _required.issubset(vector_index[0]), f"❌ Each entry needs the keys {sorted(_required)}."
#     assert vector_index[0]["type"] == "image", "❌ Image entries should carry type 'image'."
#     assert isinstance(vector_index[0]["content"], bytes) and vector_index[0]["content"], "❌ content should hold the raw image bytes."
#     assert vector_index[0]["embedding"] is not None and vector_index[0]["embedding"].shape[0] == 3072, "❌ Expected 3072-dim embeddings."
#     print(f"✅ All checks passed! {len(vector_index)} images indexed in the shared embedding space.")
# except AssertionError as e:
#     print(e)
#     print("💡 Tip: embed the raw bytes, then zip paths, bytes, and embeddings together.")

Now let's define a function that finds `top_k` most similar items from the vector_index based on a user query:

### Exercise 5: Search the index across modalities

One search function, any query modality: text finds images, images find images, and later, text finds PDFs. The shared embedding space does the heavy lifting, you do the ranking.

**Learning goal:** Implement cross-modal semantic retrieval with cosine similarity.

**What you need to implement:**

1. Embed the query: a (bytes, mime type) tuple gets embedded as binary content, a plain string as text (print what is being embedded, matching the emoji log style used elsewhere)
2. If embedding failed (`None`), report it and return an empty list
3. Collect every indexed embedding and compute cosine similarities against the query vector, flattened to one score per item
4. Take the indices of the `top_k` highest scores and return those index entries, each with an added `similarity` score, best first

**Key concepts:**

- `cosine_similarity([query_embedding], embeddings)` from sklearn returns a 2D array, `.flatten()` makes it one score per indexed item
- `np.argsort` sorts ascending, a reverse slice turns it into best-first
- Returning `{**item, "similarity": score}` keeps the original entry intact while attaching the score

**Expected output:** the demo below the function prints the query embedding log, then the transformer-architecture query returns the transformer diagram page with its similarity, and the kitten query returns the kitten photo.

**Implementation hints:**

- Handle the tuple-vs-string branch first, everything after it is modality-agnostic
- The demo cells below print "No results found" while the placeholder returns an empty list, and the cross-modal query loop in section 4.2 indexes into the result so it errors until this works

In [ ]:
# === Exercise cell: fill in the gaps below ===

def search_multimodal(
    query: str | tuple[bytes, str],
    vector_index: list[dict],
    top_k: int = 3,
) -> list[Any]:
    """
    Search the multimodal vector index for items most similar to a query.

    The query can be **either text or raw bytes** (image / PDF) — both are
    embedded with the same `gemini-embedding-2` model and compared directly
    against indexed embeddings in the shared embedding space. This gives us
    text→image, image→image, image→PDF and any other cross-modal retrieval
    without changing the index.

    Args:
        query: Text string, OR a `(bytes, mime_type)` tuple for image / PDF
            queries.
        vector_index: List of indexed items (each with an `embedding` field).
        top_k: Number of top results to return. Defaults to 3.

    Returns:
        list[Any]: List of indexed items with an added `similarity` score,
            sorted by relevance (highest first).

    Steps to complete:
    1. Embed the query (bytes tuple as binary with its mime type, plain string
       as text), logging what is being embedded
    2. If the embedding is None, report the failure and return an empty list
    3. Compute cosine similarities between the query vector and every indexed
       embedding, flattened to one score per item
    4. Return the top_k entries by similarity, each with an added
       "similarity" score, highest first
    """
    # Your implementation goes here

    return []  # Replace with the ranked result list

Let's test this with an example:

In [ ]:
query = "what is the architecture of the transformer neural network?"
results = search_multimodal(query, vector_index, top_k=1)

if not results:
    pretty_print.wrapped("❌ No results found", title="❌")
else:
    result = results[0]

    pretty_print.wrapped(
        [
            f"Similarity {result['similarity']:.3f}",
            f"Filename {result['filename']}",
        ],
        title=f"Results for query = {query}",
    )
    display_image(Path(result["filename"]))

...and another example:

In [ ]:
query = "a kitten with a robot"
results = search_multimodal(query, vector_index, top_k=1)

if not results:
    pretty_print.wrapped("❌ No results found", title="❌")
else:
    result = results[0]

    pretty_print.wrapped(
        [
            f"Similarity {result['similarity']:.3f}",
            f"Filename {result['filename']}",
        ],
        title=f"Results for query = {query}",
    )
    display_image(Path(result["filename"]))

### Validation check - run this after your implementation

Uncomment the cell below and run it after the two demo queries above. It inspects the last `results`, no API calls.

In [ ]:
# # Validation: check your Exercise 5 implementation
# try:
#     assert results, "❌ 'results' is empty. Implement search_multimodal and re-run the demo query above."
#     assert "similarity" in results[0], "❌ Each result should carry an added 'similarity' score."
#     _scores = [r["similarity"] for r in results]
#     assert all(a >= b for a, b in zip(_scores, _scores[1:])), "❌ Results should be sorted best-first."
#     assert {"content", "embedding"}.issubset(results[0]), "❌ Results should be the original index entries plus the score."
#     print("✅ All checks passed! Cross-modal search is working.")
# except AssertionError as e:
#     print(e)
#     print("💡 Tip: attach the score with {**item, 'similarity': score} and sort by descending similarity.")

## 4. Going Deeper with Multimodal Embeddings

Because every modality exists in the **same** vector space, a few patterns open up that aren't possible with text-only embedding models:

1. **Reverse search** — query the index *with an image* (or a PDF) instead of text.
2. **Truly multimodal index** — text snippets, images and PDFs interleaved in one index, all reachable from a single text query.
3. **Matryoshka dimensions** — ask the API for a smaller embedding (768, 1536, …) and get a usable subspace of the full 3072 vector (handy when storage or ANN latency matter).

### 4.1 Image-as-Query (Reverse Search)

Because text and image embeddings live in the same space, we can flip the query side: feed an **image** into the same `search_multimodal` function and retrieve the most visually similar items from the index. Below we use `image_1.jpeg` (the kitten + robot photo) as the query — the index should return the same image first (sanity check), then the next most-similar indexed item:

In [ ]:
query_image_path = Path("images") / "image_1.jpeg"
query_bytes = cast(bytes, load_image_as_bytes(query_image_path, format="WEBP", return_size=False))

results = search_multimodal((query_bytes, "image/webp"), vector_index, top_k=3)
for r in results:
    pretty_print.wrapped(f"similarity={r['similarity']:.3f}  type={r['type']:<5} filename={r['filename']}")

### 4.2 A Truly Multimodal Index: Text + Images + PDFs

Until now the index has held only image bytes. Because `gemini-embedding-2` embeds **any** modality into the same space, we can extend the existing `vector_index` with text snippets and a raw PDF — and a single text query will retrieve across all three modalities transparently.

First, two tiny helpers to add items of any type to an existing index:

In [ ]:
def add_text_to_index(vector_index: list[dict], text: str, title: str) -> None:
    """Embed a text snippet and append it to the vector index."""

    embedding = embed(text)
    vector_index.append(
        {
            "content": text,
            "type": "text",
            "title": title,
            "filename": None,
            "embedding": embedding,
        }
    )


def add_pdf_to_index(vector_index: list[dict], pdf_path: Path) -> None:
    """Embed a PDF directly (as `application/pdf`) and append it to the index.

    Note: `gemini-embedding-2` accepts up to 6 PDF pages per request. For
    longer documents you would chunk page-wise; we keep it simple here.
    """

    pdf_bytes = pdf_path.read_bytes()
    embedding = embed(pdf_bytes, mime_type="application/pdf")
    vector_index.append(
        {
            "content": pdf_bytes,
            "type": "pdf",
            "filename": pdf_path,
            "mime_type": "application/pdf",
            "embedding": embedding,
        }
    )

In [ ]:
add_text_to_index(
    vector_index,
    text=(
        "The Pacific Ocean is the largest and deepest of Earth's five oceans, "
        "covering roughly 63 million square miles. It contains the deepest "
        "known point on the planet, the Mariana Trench, which descends to "
        "nearly 11,000 metres."
    ),
    title="Pacific Ocean facts",
)
add_text_to_index(
    vector_index,
    text=(
        "A traditional French baguette is a long, thin loaf of bread made from "
        "a basic dough of flour, water, salt and yeast. It is characterised by "
        "its crisp golden crust and chewy interior, and is typically baked daily."
    ),
    title="French baguette description",
)
add_pdf_to_index(vector_index, Path("pdfs") / "attention_is_all_you_need_paper.pdf")

Summarise the resulting unified index

In [ ]:
from collections import Counter

type_counts = Counter(item["type"] for item in vector_index)
pretty_print.wrapped(
    [f"{t}: {n}" for t, n in type_counts.items()] + [f"total: {len(vector_index)}"],
    title="Unified multimodal index",
)

Now a single text query traverses **all three modalities** in one shot. We run a few queries that should each match a different modality:

In [ ]:
import textwrap

cross_modal_queries = [
    # text query -> PDF (the raw paper bytes)
    "Vaswani et al 2017 paper",
    # text query -> text snippet
    "what is the deepest ocean on Earth?",
    "a long thin loaf of crusty french bread",
    # text query -> image
    "a fluffy kitten sitting on a robot",
    "a diagram of the Transformer encoder-decoder architecture",
]

hit_paths: list[Path] = []
hit_captions: list[str] = []
for q in cross_modal_queries:
    top = search_multimodal(q, vector_index, top_k=1)[0]
    label = top.get("title") or top.get("filename")
    pretty_print.wrapped(f"Q: {q!r}\n     -> [{top['type']:<5}] {label}  (sim={top['similarity']:.3f})")
    if top["type"] == "image":
        hit_paths.append(top["filename"])
        hit_captions.append(f"{textwrap.fill(q, 30)}\n(sim={top['similarity']:.3f})")

display_image_grid(hit_paths, titles=hit_captions)

**A note for production: cross-modal score calibration.** In practice you'll notice that text↔text cosine similarities tend to sit in a different range than text↔image or text↔PDF similarities (text-text scores are usually higher and more discriminative). A single global cosine threshold therefore biases retrieval toward whichever modality dominates your corpus. Three common fixes in production:

1. **Per-modality score normalisation** — z-score or min-max scale similarities within each modality bucket before merging.
2. **Modality-aware re-ranking** — retrieve top-K per modality, then re-rank with a cross-encoder (e.g. a small Gemini call that scores each candidate against the query directly).
3. **Hybrid scoring** — combine dense similarity with a sparse signal (BM25 over OCRed image text, filename keywords, PDF metadata) to break ties when cross-modal scores cluster.

These aren't unique to Gemini. Every multimodal embedder exhibits some modality bias. But they are the patterns you'll reach for once a naive top-K starts disappointing.

### 4.3 Matryoshka: Flexible Output Dimensionality to Reduce Costs and Latency

`gemini-embedding-2` is trained with **Matryoshka Representation Learning** (MRL), which means you can ask for a smaller embedding (down to 128 dims) and still get a usable subspace of the full 3072-dim vector. In production this trades a small accuracy hit for ~4× cheaper storage and faster ANN lookups.

Below we re-build a 768-dim index over the same images and run the same search to confirm retrieval quality is preserved:

In [ ]:
DIM_SMALL = 768

# Re-embed the same images at 768 dims
small_image_paths = list(Path("images").glob("*.jpeg"))
small_image_bytes = [cast(bytes, load_image_as_bytes(p, format="WEBP", return_size=False)) for p in small_image_paths]
small_embeddings = await embed_batch(
    [(b, "image/webp") for b in small_image_bytes],
    output_dimensionality=DIM_SMALL,
)
small_index = [
    {"content": b, "type": "image", "filename": p, "mime_type": "image/webp", "embedding": e}
    for p, b, e in zip(small_image_paths, small_image_bytes, small_embeddings)
]

pretty_print.wrapped(
    f"Built {len(small_index)} items at dim={small_index[0]['embedding'].shape[0]}.\nStorage savings vs 3072 dims: ~{(1 - DIM_SMALL / 3072) * 100:.0f}%"
)

# Same query, smaller embeddings
query = "what is the architecture of the transformer neural network?"
query_emb = embed(query, output_dimensionality=DIM_SMALL)
sims = cosine_similarity(
    [query_emb],
    [d["embedding"] for d in small_index],
).flatten()
top = int(np.argsort(sims)[::-1][0])
pretty_print.wrapped(f"Top-1 @ 768 dims for {query!r}:\n  -> {small_index[top]['filename']}  (sim={sims[top]:.3f})")

Result: the Transformer page is still the top hit. In a real system you would benchmark recall@k at several dims (e.g. 256 / 768 / 1536 / 3072) on a held-out query set and pick the smallest that meets your quality bar. The same `vector_index` data structure works at every dim — only the storage and similarity-compute cost change.

## 5. Building a Multimodal RAG Agent

The last step is to hook our RAG `search_multimodal` function to a ReAct agent to create an agentic RAG system.

First, we define the `multimodal_search_tool` using LangGraph:

In [ ]:
from langchain.agents import create_agent
from langchain_core.tools import tool
from langchain_google_genai import ChatGoogleGenerativeAI


@tool
def multimodal_search_tool(query: str, top_k: int = 3) -> dict[str, Any]:
    """
    Search the multimodal vector index (text + images + PDFs) and return the
    top-`top_k` candidates so the agent can pick the most relevant one.

    Args:
        query: Text query describing what to search for (e.g., "kitten with a
            robot", "transformer architecture", "the largest ocean").
        top_k: Number of top candidates to return from the search.

    Returns:
        A tool result whose `content` interleaves a label for each candidate
        with the candidate itself — text snippet, image `Part` or PDF `Part`.
        Sorted from most-similar to least-similar.
    """

    pretty_print.wrapped(query, title="🔍 Tool executing search for:")

    results = search_multimodal(query, vector_index, top_k=top_k)

    if not results:
        return {"role": "tool_result", "content": "No relevant content found for your query."}

    summary = ", ".join(
        f"#{i + 1} [{r['type']}] {r.get('title') or r.get('filename')} (sim={r['similarity']:.3f})" for i, r in enumerate(results)
    )
    pretty_print.wrapped(summary, title=f"🔍 Found {len(results)} candidates:")

    content: list = []
    for rank, result in enumerate(results, start=1):
        item_type = result.get("type", "image")
        label = result.get("filename") or result.get("title") or "(no name)"
        header = f"Candidate #{rank} (type={item_type}, similarity={result['similarity']:.3f}): {label}"

        if item_type == "text":
            content.append({"type": "text", "text": f"{header}\n{result['content']}"})
        elif item_type == "pdf":
            content.append({"type": "text", "text": header})
            content.append(types.Part.from_bytes(data=result["content"], mime_type="application/pdf"))
        else:  # image
            content.append({"type": "text", "text": header})
            content.append(
                types.Part.from_bytes(
                    data=result["content"],
                    mime_type=result.get("mime_type", "image/jpeg"),
                )
            )

    return {"role": "tool_result", "content": content}

Next, we create a ReAct agent using LangChain's `create_agent` function and the RAG tool defined above:

### Exercise 6: Assemble the multimodal ReAct agent

The retrieval tool and its carefully written system prompt are given. The final step is wiring them into a LangChain ReAct agent backed by a Gemini chat model.

**Learning goal:** Compose a tool, a system prompt, and a chat model into a working agent.

**What you need to implement:**

1. Put `multimodal_search_tool` into the agent's tools list
2. Create the agent with `create_agent`, backed by a `ChatGoogleGenerativeAI` chat model using the more capable `gemini-2.5-pro` with a low temperature (around 0.1), plus the tools and the given system prompt
3. Return the agent

**Key concepts:**

- `create_agent` builds a LangGraph-style ReAct loop for you, no manual Thought/Action orchestration this time (compare with Lesson 8)
- The system prompt encodes the cross-modal score-bias guidance from section 4.2, read it, it is doing real work
- A stronger model pays off here because the agent must re-rank candidates across modalities

**Expected output:** the test cells below show the agent calling the search tool, inspecting candidates, and answering: the kitten's color from the photo, the Mariana Trench from the text snippet, and the paper summary from the PDF.

**Implementation hints:**

- Three short lines of wiring, the intelligence lives in the tool and the prompt
- While this returns None, the test cells below print an error from their try/except blocks, that is expected

In [ ]:
# === Exercise cell: fill in the gaps below ===

def build_react_agent() -> Any:
    """
    Build a ReAct agent with multimodal search capabilities.

    The agent retrieves from a unified index of text, images and PDFs and
    receives the top-K candidates back as Gemini `Part`s so it can re-rank
    them (helpful when cross-modal cosine scores are biased — see §4.2).

    Returns:
        Any: A LangGraph ReAct agent instance configured with multimodal search tools

    Steps to complete:
    1. Put multimodal_search_tool into the tools list
    2. Create the agent with create_agent: a ChatGoogleGenerativeAI chat
       model (gemini-2.5-pro, low temperature), the tools, and the given
       system prompt
    3. Return the agent
    """

    system_prompt = """You are a multimodal retrieval assistant with access to a single
    unified index of text snippets, images and PDF documents.

    Your behaviour for ANY user question — visual, factual, document, "what is X?":

    1. **Always call `multimodal_search_tool` first**, before answering. Do not ask
       the user for clarification or for them to upload anything — the index
       already contains the relevant material.
    2. The tool returns the top candidates ranked by cosine similarity. **Cross-modal
       similarity scores can be biased** (text↔text scores often run higher than
       text↔image), so do not blindly take rank #1. Inspect ALL returned candidates
       and pick the one that actually answers the question:
         - For "what colour / what does X look like" questions, prefer the image candidate.
         - For factual / definitional questions, prefer the text or PDF candidate.
         - If candidates from different modalities all look relevant, combine them.
    3. Only say "I don't know" AFTER inspecting the candidates and confirming none
       answers the question.

    Be concise and factual. Never refuse on the assumption that you lack information —
    you have a search tool with multiple candidates per call; use it.
    """

    # Your implementation goes here

    return None  # Replace with the assembled agent


In [ ]:
react_agent = build_react_agent()
react_agent

Now, let's test it and make the ReAct agent find the color of our kitten from the indexed dataset:

In [ ]:
try:
    test_question = "what color is my kitten?"
    pretty_print.wrapped(test_question, title="🧪 Asking question:")

    response = react_agent.invoke(input={"messages": test_question})
    messages = response.get("messages", [])
    if messages:
        final = messages[-1].content
        if isinstance(final, list):
            final = "".join(p.get("text", "") for p in final if isinstance(p, dict))
    else:
        final = "No response from the agent"
    pretty_print.wrapped(final, title="🤖 Agent response")
except Exception as e:
    print(f"❌ Error in ReAct agent: {e}")

...and a second test:

In [ ]:
try:
    test_question = "What is the deepest point on Earth, and how deep is it?"
    pretty_print.wrapped(test_question, title="🧪 Asking question:")

    response = react_agent.invoke(input={"messages": test_question})
    messages = response.get("messages", [])
    if messages:
        final = messages[-1].content
        if isinstance(final, list):
            final = "".join(p.get("text", "") for p in final if isinstance(p, dict))
    else:
        final = "No response from the agent"
    pretty_print.wrapped(final, title="🤖 Agent response")
except Exception as e:
    print(f"❌ Error in ReAct agent: {e}")

...and a third test that should retrieve the **PDF** from the index and summarise it directly:

In [ ]:
try:
    test_question = "Summarise the Vaswani et al 2017 paper in one paragraph."
    pretty_print.wrapped(test_question, title="🧪 Asking question:")

    response = react_agent.invoke(input={"messages": test_question})
    messages = response.get("messages", [])
    if messages:
        final = messages[-1].content
        if isinstance(final, list):
            final = "".join(p.get("text", "") for p in final if isinstance(p, dict))
    else:
        final = "No response from the agent"
    pretty_print.wrapped(final, title="🤖 Agent response")
except Exception as e:
    print(f"❌ Error in ReAct agent: {e}")

So far every question has been **text**. But because the underlying model is multimodal, we can also hand the agent an **image** straight in the user message (LangChain's standard `image` content block) and let it reason over the raw pixels before calling the retrieval tool.

First, a small helper that recovers the files the search tool returned from the agent's tool messages and renders the images (and lists any PDFs):

In [ ]:
import re


def get_retrieved_paths(response: dict) -> list[Path]:
    """Recover the file paths the search tool returned, by scanning the agent's tool messages."""
    paths: list[Path] = []
    for msg in response.get("messages", []):
        if getattr(msg, "type", None) != "tool":
            continue
        for match in re.findall(r"(?:images|pdfs)/[\w./-]+\.(?:jpe?g|png|webp|pdf)", str(getattr(msg, "content", ""))):
            p = Path(match)
            if p.exists() and p not in paths:
                paths.append(p)
    return paths


def render_retrieved(paths: list[Path]) -> None:
    """Render retrieved images in a grid; list any non-image files (e.g. PDFs)."""
    image_paths = [p for p in paths if p.suffix.lower() in {".jpeg", ".jpg", ".png", ".webp"}]
    other_paths = [p for p in paths if p not in image_paths]

    display_image_grid(image_paths, titles=[p.name for p in image_paths])

    for p in other_paths:
        print(f"📄 Also retrieved (not rendered inline): {p}")

**Image input → related images.** We pass a photo and ask the agent to retrieve and explain the *related* images from the index (explicitly telling it **not** to describe the input picture itself):

In [ ]:
try:
    image_path = Path("images") / "image_1.jpeg"
    image_base64 = load_image_as_base64(image_path, format="JPEG")
    question = (
        "Search my knowledge base for images related to the picture I'm sharing, then explain "
        "what those retrieved images show. Do NOT describe the picture I gave you — focus only "
        "on the images you retrieve."
    )
    pretty_print.wrapped(f"[image: {image_path}] {question}", title="🧪 Asking with an image input:")

    message = {
        "role": "user",
        "content": [
            {"type": "text", "text": question},
            {"type": "image", "base64": image_base64, "mime_type": "image/jpeg"},
        ],
    }
    response = react_agent.invoke(input={"messages": [message]})
    messages = response.get("messages", [])
    final = messages[-1].content if messages else "No response from the agent"
    if isinstance(final, list):
        final = "".join(p.get("text", "") for p in final if isinstance(p, dict))
    pretty_print.wrapped(final, title="🤖 Agent response")

    render_retrieved(get_retrieved_paths(response))
except Exception as e:
    print(f"❌ Error in ReAct agent: {e}")

**Image input → the rest of a document.** We pass a single page of the *Attention Is All You Need* paper and ask the agent to pull back everything related — the paper's other pages and the full PDF — and report what it retrieved.

In [ ]:
try:
    image_path = Path("images") / "attention_is_all_you_need_0.jpeg"
    image_base64 = load_image_as_base64(image_path, format="JPEG")
    question = (
        "Here's a page from a paper. Find every related item in my knowledge base — the other "
        "pages of the same paper and the full source document — and tell me exactly what you retrieved."
    )
    pretty_print.wrapped(f"[image: {image_path}] {question}", title="🧪 Asking with an image input:")

    message = {
        "role": "user",
        "content": [
            {"type": "text", "text": question},
            {"type": "image", "base64": image_base64, "mime_type": "image/jpeg"},
        ],
    }
    response = react_agent.invoke(input={"messages": [message]})
    messages = response.get("messages", [])
    final = messages[-1].content if messages else "No response from the agent"
    if isinstance(final, list):
        final = "".join(p.get("text", "") for p in final if isinstance(p, dict))
    pretty_print.wrapped(final, title="🤖 Agent response")

    render_retrieved(get_retrieved_paths(response))
except Exception as e:
    print(f"❌ Error in ReAct agent: {e}")

Above, the agent answered **five** questions. The first three were **text** queries, each retrieving from a different modality of the unified index. The last two used an **image as the input**: the model looked at the picture, searched the same unified index, and we rendered the items it pulled back. Related photos in one case, and the rest of a paper's pages plus the full PDF in the other. Same tool, same index, same embedding model.

## Stretch challenges

Want to go further? Try these on your own:

1. Query the index with the PDF itself: load the paper's bytes and pass a `(bytes, "application/pdf")` tuple to your `search_multimodal`, then inspect which pages and snippets come back.
2. Rebuild the index at 256 dimensions with the Matryoshka `output_dimensionality` option and measure how often the top-1 result changes across the five cross-modal queries from section 4.2.
3. Add per-modality score normalization to `search_multimodal` (z-score the similarities within each type bucket before ranking) and compare rankings on the cross-modal queries, this implements fix #1 from the production note above.